In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, KFold
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
%matplotlib inline

# PreProcessing Data Cleaning (Only Once)
- output: NFI7-임목조사표-filtered3-training.csv

In [ ]:
parent_dir = r"D:/ForestFire/CBH"
df = pd.read_csv(os.path.join(parent_dir, r'data/NFI7-Immok-Filtered3.csv'))
df.info()

In [ ]:
# 임상도에 따라 NFI 수종명 재분류
nfi_names = df['수종명'].unique()
nfi_imsang = [df.loc[(df['수종명']==name), '침활구분'].unique()[0] for name in nfi_names]
nfi_dict = {i : j for i, j in zip(nfi_names, nfi_imsang)}
# 임상도 수종 및 코드 분류에 따라 수종코드(SID) 부여하기
code_species_dict = {11: ['소나무'], 12:['잣나무', '섬잣나무', '눈잣나무', '스트로브잣나무'], 13: ['일본잎갈나무', '잎갈나무'], 14: ['리기다소나무', '리기테다소나무', '방크스소나무'],
                    15: ['곰솔'], 16: ['전나무', '구상나무', '분비나무'], 17: ['편백', '화백'], 18: ['삼나무', '낙우송','메타세콰이아'], 19: ['가문비나무', '독일가문비나무', '종비나무'],
                    20: ['비자나무', '개비자나무'], 21: ['은행나무'], 31: ['상수리나무'], 32: ['신갈나무'], 33: ['굴참나무'], 34: ['갈찬나무', '떡갈나무', '졸참나무'],
                    35: ['오리나무', '물오리나무', '사방오리'], 36: ['고로쇠나무'], 37: ['자작나무', '거제수나무'],  38: ['박달나무', '개박달나무', '물박달나무'], 39: ['밤나무'],
                    40: ['물푸레나무', '들메나무', '물들메나무'], 41: ['서어나무', '개서어나무'], 42: ['때죽나무', '쪽동백나무'], 43: ['호두나무', '가래나무'], 44:['백합나무'], 
                    45: ['미루나무', '은사시나무', '이태리포플러나무', '수원사시나무'], 46: ['벚나무', '양벚나무', '산벚나무', '꽃벚나무', '왕벚나무'], 47: ['느티나무'],  48:['층층나무', '곰의말채나무'],
                    49: ['아까시나무'], 61: ['가시나무', '붉가시나무', '종가시나무', '참가시나무', '개가시나무'], 62: ['구실잣밤나무'], 63: ['녹나무'], 64: ['굴거리나무'], 65: ['황칠나무'], 66: ['사스레피나무'], 67: ['후박나무'],
                     68: ['새덕이', '참식나무', '생달나무']}
id_lst = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 61, 62, 63, 64, 65, 66, 67, 68]
name_lst1 = ['소나무', '잣나무', '낙엽송', '리기다소나무', '곰솔', '전나무', '편백나무', '삼나무', '가문비나무', '비자나무', '은행나무', '상수리나무', '신갈나무', '굴참나무', '기타참나무류', '오리나무', '고로쇠나무', '자작나무', '박달나무', '밤나무', '물푸레나무', '서어나무', '때죽나무', '호두나무', '백합나무', '포플러', ' 벚나무', 
             '느티나무', '층층나무', '아까시나무', '가시나무', '구실잣밤나무', '녹나무', '굴거리나무', '황칠나무','사스피레나무', '후박나무','새덕이']
name_lst2 = ['기타침엽수', '기타 참나무류', '기타활엽수']
code_name_dict = {i: j for i, j in zip(id_lst, name_lst1)}
len(id_lst), len(name_lst1), len(code_species_dict), len(code_name_dict)

In [ ]:
# 속성 추출 및 단위 환산
df2 = df[['표본점번호', '수종명', '침활구분', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']]
cm_to_inch = 0.3937
cm_to_ft = 0.0328084
df2['흉고직경'] = df2['흉고직경'].apply(lambda x: x * cm_to_inch)
df2['수고'] = df2['수고'].apply(lambda x: x * cm_to_ft)
df2['지하고'] = df2['지하고'].apply(lambda x: x * cm_to_ft)
df2['해발고(m)'] = df2['해발고(m)'] / 100 # hm로 변환
df2['경사(degree)'] = np.tan(np.radians(df2['경사(degree)'])) # tangent로 변환
df2['방위각(º)'] = np.radians(df2['방위각(º)']) # radian으로 변환
df2['평균수관밀도(%)'] = df2['평균수관밀도(%)'] / 100 # 소수점 자릿수로 변환

# ['표본점번호', '수종명', '흉고직경', '수고', '수령', '지하고', '해발고(m)', '경사(degree)', '방위각(º)','평균수관밀도(%)','좌표N', '좌표E']
df2.columns = ['SampleID', 'Species', 'Imsang', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long']
df2.info

In [ ]:
# 수관비율 생성
df2.insert(6, 'CR', (df2['CBH(ft)'] / df2['H(ft)']))
df2.insert(7, 'CH', (df2['H(ft)'] - df2['CBH(ft)']))

In [ ]:
# reset or add the columns
df2['I_Species'] = np.full(len(df2), '-99')
df2['SID'] = np.full(len(df2), -99)

# 임상도 기준 수종명 및 수종코드 칼럼 삽입
for key, name_lst in code_species_dict.items(): 
    condition = df2['Species'].isin(name_lst)
    df2.loc[condition, 'I_Species'] = code_name_dict[key]
    df2.loc[condition, 'SID'] = key
    
    condition2 = ((df2['Imsang'] == '활엽수') & (df2['I_Species'] == '-99'))
    df2.loc[condition2, 'I_Species'] = '기타활엽수'
    df2.loc[condition2, 'SID'] = 30

    condition3 = ((df2['Imsang'] == '침엽수') & (df2['I_Species'] == '-99'))
    df2.loc[condition3, 'I_Species'] = '기타침엽수'
    df2.loc[condition3, 'SID'] = 10

In [ ]:
df2.I_Species.unique()

In [ ]:
df_fin = pd.concat([df2[df2.columns[-2:]], df2[df2.columns[:3]], df2[df2.columns[3:15]]], axis=1)
print(df_fin.shape)
if 'Age' in df_fin.columns: df_fin = df_fin.drop(columns=['Age'])
df_fin = df_fin.dropna()
print(df_fin.shape)
data_dir = r"D:\ForestFire\CBH\data"
df_fin.to_csv(os.path.join(data_dir, 'NFI7-임목조사표-filtered3-training.csv'), encoding='cp949', index=False)

In [ ]:
# 임목 단위에서의 수종별 비율
total_count = len(df_fin)
species_name = df_fin.I_Species.unique()
species_count = []
species_ratio = []
for i in df_fin.I_Species.unique():
    count = df_fin.loc[df_fin['I_Species'] == i, 'SID'].count()
    species_count.append(count)
    species_ratio.append(count/total_count)

data_summary = pd.DataFrame({'Species' : species_name, 'Count' : species_count, 'Ratio' : species_ratio})
data_summary.to_csv(os.path.join(result_dir, 'NFI7-수종별비율.csv'))

# Visualizaiton

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

In [ ]:
# DBH, Height 정규화하기
scaler1 = StandardScaler()
robust1 = RobustScaler()
dbh_scaled = scaler1.fit_transform(df3[['DBH(inch)']])
dbh_scaled2 = robust1.fit_transform(df3[['DBH(inch)']])
dbh_log = np.log1p(df3[['DBH(inch)']])
dbh_exp = np.exp(df3[['DBH(inch)']])
scaler2 = StandardScaler()
robust2 = RobustScaler()
h_scaled = scaler2.fit_transform(df3[['H(ft)']])
h_scaled2 = robust2.fit_transform(df3[['H(ft)']])
h_log = np.log1p(df3[['H(ft)']])
h_exp = np.exp(df3[['H(ft)']])
# 분포 비교하기
fig, ax = plt.subplots(2,5, figsize=(10,5)) # Object-oriented subplot / plt.subplot(1,2,1: State-based)
# plt.grid(color='gray', linestyle='--', linewidth=.5, alpha=.5)
# graphs of DBH
ax[0,0].hist(x=df3['DBH(inch)'], bins=100)
ax[0,0].set_title('Original')
ax[0,1].hist(x=dbh_scaled, bins=100)
ax[0,1].set_title('Standard Scaler')
ax[0,2].hist(x=dbh_scaled2, bins=100)
ax[0,2].set_title('Robust Scaler')
ax[0,3].hist(x=dbh_log, bins=100)
ax[0,3].set_title('Log transform')
ax[0,4].hist(x=dbh_exp, bins=100)
ax[0,4].set_title('Exponential transform')
# graphs of Height
ax[1,0].hist(x=df3['H(ft)'], bins=100)
ax[1,0].set_title('Original')
ax[1,1].hist(x=h_scaled, bins=100)
ax[1,1].set_title('Standard Scaler')
ax[1,2].hist(x=h_scaled2, bins=100)
ax[1,2].set_title('Robust Scaler')
ax[1,3].hist(x=h_log, bins=100)
ax[1,3].set_title('Log Transform')
ax[1,4].hist(x=h_exp, bins=100)
ax[1,4].set_title('Exponential Transform')

fig.text(0.5, 1, "DBH", ha="center", fontsize=14, fontweight="bold")
fig.text(0.5, 0.51, "Height", ha="center", fontsize=14, fontweight="bold")
plt.subplots_adjust(hspace=0.5)


plt.tight_layout()
plt.show()

### EDA: Draw the scatter plot

In [ ]:
def drawPairPlot(df, s_name, opt_save, f_name=None):
    condition = (df['Species'] == s_name)
    df_species = df.loc[condition, ['DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'CR', 'CH', 'Elev(hm)', 'Azimuth(rad)', 'CD(%)']]
    sns.pairplot(df_species, kind='reg', plot_kws={'line_kws' : {'color' : 'orange'}})
    if opt_save: 
        save_dir = r'D:/ForestFire/CBH/fig'
        plt.savefig(os.path.join(save_dir, f_name))

In [ ]:
# 소나무 scatterplot
drawPairPlot(df2, '소나무', opt_save=True, f_name='소나무-Scatter.png')

In [ ]:
# 굴참나무 scatterplot
drawPairPlot(df2, '굴참나무', opt_save=True, f_name='굴참-Scatter.png')

In [ ]:
# 왕벚나무 scatterplot
drawPairPlot(df2, '왕벚나무', opt_save=True, f_name='왕벚-Scatter.png')

In [ ]:
# 고로쇠 scatterplot
drawPairPlot(df2, '고로쇠나무', opt_save=True, f_name='고로쇠-Scatter(30).png')

### Trial - Build the CCF equation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

In [ ]:
df2.Species

In [ ]:
X_train.shape

In [ ]:
tree_n = '고로쇠나무'
features = ['DBH(inch)', 'CH']
df_pinus = df2[df2['Species'] == tree_n].loc[:, features].reset_index(drop=True)
ratio = .3

df_train = df_pinus.sample(frac=(1 - ratio), replace=False)
df_test = df_pinus.sample(frac=ratio, replace=False)
X_test = df_test[['DBH(inch)']]
y_test = df_test['CH']
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)


# trian and test model
ccf_pinus = LinearRegression()

# Cross-validation for training set
kf = KFold(n_splits=3, shuffle=True, random_state=44)
best_score = -np.inf
cv_scores = []
best_coef = None
best_intercept = None
test_score = None

# perform cross-validation
X = df_train[['DBH(inch)']]
y = df_train['CH']
for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # train model
    ccf_pinus.fit(X_train, y_train)
    score = ccf_pinus.score(X_val, y_val)
    cv_scores.append(score)
    # save the best score
    if score > best_score:
        best_score = score
        best_coefs = ccf_pinus.coef_
        best_intercept = ccf_pinus.intercept_
        # test score
        test_score = ccf_pinus.score(X_test, y_test)
    
print('Best score(r2): ', best_score)
print('Test score(r2): ', test_score)
print('Best coef: ', best_coefs)
print('Best intercept: ', best_intercept)
print(cv_scores)

# Build species-dependent Model (Hansenauer & Monserud, 1996)

In [ ]:
from scipy.optimize import minimize # regularization 적용 툴
data_dir = r"D:\ForestFire\CBH\data"
file_name = r"NFI6-7_cleaned.csv"
df_all = pd.read_csv(os.path.join(data_dir, file_name), encoding='cp949')
df_all = df_all.reset_index()

In [ ]:
# 함수 정의
# Baseline function
def func(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    size = (b1 * H/D)+(b2 * H)+(b3 * D**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr
# funciton with log transformation of DBH & Height
def func2(X, a, b1, b2, b3, c1, d1, d2, d3, d4, d5, d6):
    H, D, EL, SL, AZ, CD = X
    H_log, D_log = np.log1p(H), np.log1p(D)
    size = (b1 * H_log/D_log)+(b2 * H_log)+(b3 * D_log**2)
    comp = c1 * CD
    site = (d1 * EL)+(d2 * EL**2) + (d3 * SL)+(d4 * SL**2)+(d5 * SL * np.sin(AZ))+(d6 * SL * np.cos(AZ))
    x = size + comp + site + a
    cr = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    return cr

# loss function for baseline function
def loss_func(params, lam, X, y):
    y_pred = func(X, *params)
    return np.sum((y - y_pred) ** 2) + lam * np.sum(params**2) # L2 규제 적용

# loss function for func2
def loss_func2(params, lam, X, y):
    y_pred = func2(X, *params)
    return np.sum((y - y_pred) ** 2) + lam * np.sum(params**2) # L2 규제 적용

In [ ]:
# read training dataset
df3 = pd.read_csv(os.path.join(data_dir, 'NFI7-임목조사표-filtered3-training.csv'), encoding='cp949')

In [ ]:
X

### Baseline Model

In [ ]:
# Apply CV
# record array 만들기: (species_id, data_count, (coef 11), r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)
target_trees =  df3['SID'].unique()
rec = np.zeros((len(target_trees), 19))
rec[:, 0] = target_trees
print(rec.shape)

# 'SampleID', 'Species', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long'
lam = 0.1 #0.0 - No regularization, 0.01 - Slight, 0.1 - Moderate, 1 - Overly strong
test_ratio = 0.2
n_fold = 5

for sid in tqdm(target_trees):
    condition = (df3['SID'] == sid)
    df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
    cnt = len(df_species)
    popt = [-999] * 11
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    if (df_species.isna().any(axis=1).sum()) > 0:
        print('Null data exsits. Delete or manage the null data.')
    else:
        if cnt >= 30:
            # train-test data split
            df_test = df_species.sample(frac=test_ratio, replace=False)
            df_train = df_species.drop(index=df_test.index)
            # create test X, y for curve-fit
            X_test = np.array(df_test.iloc[:, :-1]).T
            y_test = df_test.iloc[:, -1]
            X_train = df_train.iloc[:, :-1]
            y_train = df_train.iloc[:, -1]

            # train curvefit
            popt, pcov = curve_fit(func2, X_train.values.T, np.array(y_train))
            opt_params = popt
    
            # predict & evaluate performance with the best params
            y_pred = func2(X_train.values.T, *opt_params)
            score = r2_score(y_train, y_pred)

            # evaluate with the best-score params
            r2_train = r2_score(y_train, y_pred)
            mae_train = mean_absolute_error(y_train, y_pred)
            rmse_train = root_mean_squared_error(y_train, y_pred)
    
            # test with the best-score params
            y_pred2 = func2(X_test, *opt_params)
            r2_test = r2_score(y_test, y_pred2)
            mae_test = mean_absolute_error(y_test, y_pred2)
            rmse_test = root_mean_squared_error(y_test, y_pred2)
    
        # save the result in the record list
        rec[rec[:, 0] == sid, :] = [sid, cnt] + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test] + list(opt_params)
        result_dir = r'D:/ForestFire/CBH/result'
        np.savetxt(os.path.join(result_dir, 'CR_Han_result_baseline.csv'), rec, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

In [ ]:
print(r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)

In [ ]:
df_result = pd.DataFrame(rec, columns=['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'] + [f'Par{i}' for i in range(11)])
code_name_dict[10] = '기타 침엽수'
code_name_dict[30] = '기타 활엽수'
species_names2 = [code_name_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', species_names2)
df_result.to_csv(os.path.join(result_dir, 'CR_Han_result_baseline.csv'), encoding='cp949')
df_result2 = df_result[df_result['Count'] >= 30].reset_index(drop=True)
df_result2

In [ ]:
title = "CR_Han_result_baseline"
df_result2 = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')
df_result2
plt.rc('font', family='Malgun Gothic')  # Use 'Malgun Gothic' for Windows or 'AppleGothic' for Mac
plt.rcParams['axes.unicode_minus'] = False  # Ensure minus signs are displayed correctly

# Set figure size
plt.figure(figsize=(12, 6))

# Define bar width and positions
bar_width = 0.4
x = np.arange(len(df_result2["SID"]))

# Create the bars
plt.bar(x - bar_width/2, df_result2["r2_tr"], width=bar_width, label="r2_tr", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, df_result2["r2_te"], width=bar_width, label="r2_te", color="orange", alpha=0.7)

# Add count annotations
for i in range(len(df_result2)):
    plt.text(x[i] - bar_width/2, df_result2["r2_tr"][i] + 0.02, f'{df_result2["r2_tr"][i]:.2f}', ha='center', fontsize=5)
    plt.text(x[i] + bar_width/2, df_result2["r2_te"][i] + 0.02, f'{df_result2["r2_te"][i]:.2f}', ha='center', fontsize=5)

# Labels and title
x_label_list = [df_result2.loc[i, 'SName'] + f'({int(df_result2.loc[i, 'Count'])})' for i in range(len(df_result2))]
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.xticks(ticks=x, labels=x_label_list, rotation=45, ha="right")
plt.legend()

# Show plot
plt.tight_layout()
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()

### With Cross Validation + L2 Regularization

In [ ]:
df3['SID'].unique()

In [ ]:
# Apply CV
# record array 만들기: (species_id, data_count, (coef 11), r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test)
target_trees =  [37] # 사스레피나무, 새덕이 # 전 수종을 구축할 경우, df3['SID'].unique() [66, 35, 49, 62, 64, 68, 12, 44, 43, 38]
rec = np.zeros((len(target_trees), 19))
rec[:, 0] = target_trees
print(rec.shape)

# 'SampleID', 'Species', 'DBH(inch)', 'H(ft)', 'Age', 'CBH(ft)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)', 'Lat', 'Long'
lam = 0.1 #0.0 - No regularization, 0.01 - Slight, 0.1 - Moderate, 1 - Overly strong
test_ratio = 0.2
n_fold = 5

for sid in tqdm(target_trees):
    condition = (df3['SID'] == sid)
    df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
    cnt = len(df_species)
    popt = [-999] * 11
    r2_train = None
    mae_train = None
    rmse_train = None
    r2_test = None
    mae_test = None
    rmse_test = None
    if (df_species.isna().any(axis=1).sum()) > 0:
        print('Null data exsits. Delete or manage the null data.')
    else:
        if cnt >= 30:
            # train-test data split
            df_test = df_species.sample(frac=test_ratio, replace=False)
            df_train = df_species.drop(index=df_test.index)
            # create test X, y for curve-fit
            X_test = np.array(df_test.iloc[:, :-1]).T
            y_test = df_test.iloc[:, -1]
            
            # Cross-validation for training set
            kf = KFold(n_splits=n_fold, shuffle=True) # 5th: 66 # 4th: 55 # 3th: 444
            best_score = -np.inf
            cv_scores = []
            best_params = None
            test_score = None
            
            # perform cross-validation
            X = df_train.iloc[:, :-1]
            y = df_train.iloc[:, -1]
            for train_index, val_index in kf.split(X):
                X_train, X_val = X.iloc[train_index], X.iloc[val_index]
                y_train, y_val = y.iloc[train_index], y.iloc[val_index]
                
                # create train-val X-array for curve-fit
                X_train = np.array(X_train).T
                X_val = np.array(X_val).T
                
                # train curvefit
                popt, pcov = curve_fit(func2, np.array(X_train), np.array(y_train))
                
                # optimize the parameters using L2 regularization (minimize funciton)
                result_reg = minimize(loss_func2, x0=popt, args=(lam, X_train, y_train)) # (손실함수, 초기값, 그외 전달인자: lambda값, X, y)
                opt_params = result_reg.x
        
                # predict & evaluate performance with the best params
                y_pred = func2(X_train, *opt_params)
                score = r2_score(y_train, y_pred)
                # save the result of the best score
                if score > best_score:
                    best_score = score
                    best_params = opt_params
                
            # evaluate with the best-score params
            best_pred = func2(np.array(X).T, *best_params)
            r2_train = r2_score(y, best_pred)
            mae_train = mean_absolute_error(y, best_pred)
            rmse_train = root_mean_squared_error(y, best_pred)
    
            # test with the best-score params
            y_pred2 = func2(X_test, *best_params)
            r2_test = r2_score(y_test, y_pred2)
            mae_test = mean_absolute_error(y_test, y_pred2)
            rmse_test = root_mean_squared_error(y_test, y_pred2)
    
        # save the result in the record list
        rec[rec[:, 0] == sid, :] = [sid, cnt] + [r2_train, mae_train, rmse_train, r2_test, mae_test, rmse_test] + list(best_params)
        result_dir = r'D:/ForestFire/CBH/result'
        # np.savetxt(os.path.join(result_dir, 'CR_Han_result_LogTrans_CV8.4.txt'), rec, delimiter=',',fmt='%.2f', header=','.join(['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te']))

In [ ]:
df_result = pd.DataFrame(rec, columns=['SID','Count' ,'r2_tr', 'mae_tr', 'rmse_tr', 'r2_te', 'mae_te', 'rmse_te'] + [f'Par{i}' for i in range(11)])
code_name_dict[10] = '기타 침엽수'
code_name_dict[30] = '기타 활엽수'
species_names2 = [code_name_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', species_names2)
df_result.to_csv(os.path.join(result_dir, 'CR_Han_result_LogTransform_CV9.5.csv'), encoding='cp949')
df_result2 = df_result[df_result['Count'] >= 30].reset_index(drop=True)
df_result2

In [ ]:
# Set font to support Korean characters
title = "CR_Han_result_LogTransform_CV9"
df_result2 = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')
plt.rc('font', family='Malgun Gothic')  # Use 'Malgun Gothic' for Windows or 'AppleGothic' for Mac
plt.rcParams['axes.unicode_minus'] = False  # Ensure minus signs are displayed correctly

# Set figure size
plt.figure(figsize=(12, 6))

# Define bar width and positions
bar_width = 0.4
x = np.arange(len(df_result2["SID"]))

# Create the bars
plt.bar(x - bar_width/2, df_result2["r2_tr"], width=bar_width, label="r2_tr", color="blue", alpha=0.7)
plt.bar(x + bar_width/2, df_result2["r2_te"], width=bar_width, label="r2_te", color="orange", alpha=0.7)

# Add count annotations
for i in range(len(df_result2)):
    plt.text(x[i] - bar_width/2, df_result2["r2_tr"][i] + 0.02, f'{df_result2["r2_tr"][i]:.2f}', ha='center', fontsize=5)
    plt.text(x[i] + bar_width/2, df_result2["r2_te"][i] + 0.02, f'{df_result2["r2_te"][i]:.2f}', ha='center', fontsize=5)

# Labels and title
x_label_list = [df_result2.loc[i, 'SName'] + f'({int(df_result2.loc[i, 'Count'])})' for i in range(len(df_result2))]
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.xticks(ticks=x, labels=x_label_list, rotation=45, ha="right")
plt.legend()

# Show plot
plt.tight_layout()
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()

### 누락된 나무 확인

In [ ]:
no_species = ['낙엽송', '편백나무', '가문비나무', '비자나무', '호두나무', '포플러', '벚나무', '가시나무', '녹나무']
df2[df2['Species'].isin(no_species)]

### 다중공선성 확인

In [ ]:
result_dir = r'D:/ForestFire/CBH/result'

In [ ]:
data = np.loadtxt(os.path.join(result_dir, 'CR_Han_result_LogTransform_CV_FIN.csv'), skiprows=1, dtype='object', delimiter=',')
names = data[:, 1]
mask = np.ones(data.shape[1], dtype=bool)
mask[1] = False
data2 = data[:, mask].astype('float')

In [ ]:
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", 200)         # Set display width
pd.set_option("display.max_colwidth", 100)  # Set max width for individual cells

In [ ]:
# print the pcov
track = 0
param_name = ['a', 'b1', 'b2', 'b3', 'c1', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6']
file_path = os.path.join(result_dir, 'covariance.xlsx')
if not os.path.exists(file_path):
    # Create an empty DataFrame and save it to the file
    with pd.ExcelWriter(file_path, mode='w', engine='openpyxl') as writer:
        pd.DataFrame().to_excel(writer, sheet_name="Placeholder")
for sid in tqdm(df3['SID'].unique()):
    if data2[(data2[:, 0]==sid), 1] > 10:
        condition = (df3['SID'] == sid)
        df_species = df3.loc[condition, ['H(ft)', 'DBH(inch)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'CD(%)','CR']]
        X = np.array(df_species.iloc[:, :-1]).T
        y = df_species.iloc[:, -1]
        popt, pcov = curve_fit(func2, X, y, p0=data2[track, -11:])
        df_cov = pd.DataFrame(data=pcov, columns = param_name, index = param_name)
        with pd.ExcelWriter(file_path, mode='a', engine='openpyxl', if_sheet_exists='new') as writer:
            df_cov.to_excel(writer, sheet_name=f'Sheet{sid}', index=True)
        print(f"Covariance matrix of {names[track]}") 
        print(df_cov)
        track += 1

# Distribution of DBH, Height, CD

In [ ]:
data = df[['흉고직경', '수고', '평균수관밀도(%)']]

In [ ]:
%matplotlib inline

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(10, 5))
plt.title('Distribution of all trees')
for i, name in enumerate(data.columns):
    ax[i].hist(data[name])
    ax[i].set_title(name)
plt.savefig(os.path.join(fig_dir, 'distribution-alltrees.png'))

In [ ]:
# 구간별 분포 - 수관밀도
threshold = [0, 50, 70, 100]
fig, ax = plt.subplots(1,3, figsize=(10, 5))
plt.title('Distribution of crown density')
for i in range(len(threshold)-1):
    condition = (data['평균수관밀도(%)'] > threshold[i]) & (data['평균수관밀도(%)'] <= threshold[i+1])
    df_cr = data.loc[condition, '평균수관밀도(%)']
    ax[i].hist(df_cr, bins=30)
    ax[i].set_title(f'Crown Density({threshold[i]}-{threshold[i+1]}%)')

In [ ]:
# assume and fit a distribution
# 0-50: gamma distribution or log-normal distribtuijon
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

def find_best_fit_distribution(data, distributions):
    """Finds the best-fitting distribution using the KS test and returns results."""
    results = {}
    x = np.linspace(data.min(), data.max(), 1000)

    for name, dist in distributions.items():
        try:
            # Fit distribution to data
            params = dist.fit(data)
            pdf_fitted = dist.pdf(x, *params)
            ks_stat, ks_pval = stats.kstest(data, lambda x: dist.cdf(x, *params))

            # Store results
            results[name] = {
                "params": params,
                "KS Statistic": ks_stat,
                "p-value": ks_pval,
                "pdf": pdf_fitted
            }
        except Exception as e:
            print(f"Skipping {name} due to error: {e}")

    # Select best fit (highest p-value)
    best_fit = max(results, key=lambda d: results[d]["p-value"])
    
    return best_fit, results

# Load sample data (replace with your actual dataset

# Define probability distributions to test
distributions = {
    "Gamma": stats.gamma,
    "Log-Normal": stats.lognorm,
    "Beta": stats.beta,
    "Weibull": stats.weibull_min,
    "Exponential": stats.expon,
    "GEV" : stats.genextreme
}

# Find the best-fitting distribution
attribute = '평균수관밀도(%)'
lower_bound = 0
upper_bound = 4000
condition = (data[attribute] > lower_bound) & (data[attribute] <= upper_bound)
data_cd = data.loc[condition,attribute].dropna() # condition
best_fit, results = find_best_fit_distribution(data_cd, distributions)

# Plot histogram and best fit
x = np.linspace(data_cd.min(), data_cd.max(), len(results[best_fit]["pdf"]))
plt.figure(figsize=(8, 5))
plt.hist(data_cd, bins=30, density=True, color='gray', alpha=0.6, label="Data Histogram")
plt.plot(x, results[best_fit]["pdf"], label=f"Best Fit: {best_fit}", linewidth=2, color="red")
plt.xlabel("Crown Density (%)")
plt.ylabel("Probability Density")
plt.title("Best-Fitting Probability Distribution for DBH")
plt.legend()
plt.show()

# Print best fit results
print(f"Best-Fitting Distribution: {best_fit}")
print(f"Parameters: {results[best_fit]['params']}")
print(f"KS Statistic: {results[best_fit]['KS Statistic']}")
print(f"P-Value: {results[best_fit]['p-value']}")

In [ ]:
max(data['수고(m)']), max(data['흉고직경'])

In [ ]:
# Define thresholds
threshold = [0] + list(np.arange(1, 41, 2)) + [4000]  # len(threshold) = 22
data['수고(m)'] = data['수고'].apply(lambda x: x/100)
# Create 5x5 subplot grid
fig, ax = plt.subplots(5, 5, figsize=(15, 12))  # Increased figure size
fig.suptitle('Distribution of Tree Height', fontsize=16)  # Set overall title

num_bins = len(threshold) - 1  # 21 bins

for idx in range(num_bins):
    i, j = divmod(idx, 5)  # Get subplot row and column index # 몫과 나머지의 개념으로!
    lower, upper = threshold[idx], threshold[idx+1]

    # Filter data for the given height range
    condition = (data['수고(m)'] > lower) & (data['수고(m)'] <= upper)
    df_cr = data.loc[condition, '수고(m)']
    cnt = len(df_cr)

    # Plot histogram in the corresponding subplot
    if cnt < 100: bin_n = 10
    else: bin_n = 20
    ax[i, j].hist(df_cr, bins=bin_n, color='skyblue', edgecolor='black')
    ax[i, j].set_title(f'{lower}-{upper} m ({cnt})', fontsize=10)
    ax[i, j].tick_params(axis='both', which='major', labelsize=8)

# Remove empty subplots (if any)
for idx in range(num_bins, 25):
    fig.delaxes(ax.flatten()[idx])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
plt.savefig(os.path.join(fig_dir, 'distribution-treeHeight.png'))
plt.show()

In [ ]:
max(data['흉고직경'])

In [ ]:
# 경급
threshold = [0, 6, 18, 30, (max(data['흉고직경'])+1)]
# Create 5x5 subplot grid
fig, ax = plt.subplots(1, 4, figsize=(12, 4)) 
fig.suptitle('Distribution of Tree DBH', fontsize=16)  # Set overall title

num_bins = len(threshold) - 1  # 4 bins

for idx in range(num_bins):
    lower, upper = threshold[idx], threshold[idx+1]
    # Filter data for the given height range
    condition = (data['흉고직경'] > lower) & (data['흉고직경'] <= upper)
    df_cr = data.loc[condition, '수고(m)']
    cnt = len(df_cr)

    # Plot histogram in the corresponding subplot
    if cnt < 100: bin_n = 10
    else: bin_n = 20
    ax[idx].hist(df_cr, bins=bin_n, color='skyblue', edgecolor='black')
    ax[idx].set_title(f'{lower}-{upper} m ({cnt})', fontsize=10)
    ax[idx].tick_params(axis='both', which='major', labelsize=8)


plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
plt.show()